In [ ]:
#References: https://medium.com/@ayush.rane/experimenting-with-ml-trying-out-different-algorithms-for-one-simple-task-2431369c0223

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

In [2]:
# Load dataset
df = pd.read_csv("../data/merged_final_transformed.csv")
print(f'Shape: {df.shape}')
df.head()


Shape: (6646, 41)


,year,StateAbbr,County name,CountyFIPS,BPHIGH,CASTHMA,COPD,MHLTH,PHLTH,SLEEP,...,median_age,pct_bachelors_plus,pct_graduate_degree,pct_less_than_hs,pct_white,pct_black,pct_asian,pct_hispanic,median_household_income,climate_type_short
0,2013,AK,ANCHORAGE MUNICIPALITY,2020,28.691667,NaN,NaN,NaN,NaN,NaN,...,32.8,11.7,92.5,23.2,0.1,7.9,7.8,92.0,77454,ET
1,2013,AL,JEFFERSON COUNTY,1073,41.893750,NaN,NaN,NaN,NaN,NaN,...,37.2,11.6,87.4,26.6,0.1,0.8,0.8,96.2,45429,Cfa
2,2013,AL,MADISON COUNTY,1089,37.654348,NaN,NaN,NaN,NaN,NaN,...,37.4,14.5,90.0,21.4,0.1,2.3,2.2,95.4,58434,Cfa
3,2013,AL,MOBILE COUNTY,1097,44.833333,NaN,NaN,NaN,NaN,NaN,...,36.7,7.0,83.9,33.0,0.0,1.4,1.4,97.5,43028,Cfa
4,2013,AL,MONTGOMERY COUNTY,1101,39.792727,NaN,NaN,NaN,NaN,NaN,...,34.8,12.0,85.6,26.2,0.1,1.3,1.1,96.5,44790,Cfa


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6646 entries, 0 to 6645
Data columns (total 41 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   year                     6646 non-null   int64  
 1   StateAbbr                6646 non-null   str    
 2   County name              6646 non-null   str    
 3   CountyFIPS               6646 non-null   int64  
 4   BPHIGH                   6031 non-null   float64
 5   CASTHMA                  6322 non-null   float64
 6   COPD                     6322 non-null   float64
 7   MHLTH                    6322 non-null   float64
 8   PHLTH                    6322 non-null   float64
 9   SLEEP                    3195 non-null   float64
 10  STROKE                   6322 non-null   float64
 11  STATION                  6601 non-null   str    
 12  STATION_NAME             6601 non-null   str    
 13  LATITUDE                 6601 non-null   float64
 14  LONGITUDE                6601 non-n

In [4]:
missing = df.isnull().sum() / len(df) * 100
missing = missing[missing > 0].sort_values(ascending=False)
print('Features with missing values (% missing):')
print(missing.round(1).to_string())

Features with missing values (% missing):
SLEEP           51.9
BPHIGH           9.3
PRCP             8.5
HTDD             8.0
CLDD             7.8
TAVG             7.7
TMIN             7.4
DT32             7.4
EMNT             7.4
TMAX             7.3
EMXT             7.3
DX90             7.3
DX70             7.3
DX32             7.3
STROKE           4.9
CASTHMA          4.9
MHLTH            4.9
COPD             4.9
PHLTH            4.9
ELEVATION        0.7
LONGITUDE        0.7
LATITUDE         0.7
STATION_NAME     0.7
STATION          0.7


# Preparing the data with one hat encoding categorical 

In [5]:
# Separate feature types — keep as lists for ColumnTransformer
target = "CASTHMA"

df = df.drop(columns=["County name", "CountyFIPS", 'BPHIGH', 'SLEEP', 'COPD', 'MHLTH', 'PHLTH','STROKE'])
#Should we drop STATION? STATION_NAME?
categorical_features = df.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
numerical_features = [c for c in df.select_dtypes(include=['float64', 'int64']).columns
                      if c != target]

The datasets for targets != SLEEP are larger due to dropping sleep columns that have missing values. 

In [6]:
missing = df.isnull().sum() / len(df) * 100
missing = missing[missing > 0].sort_values(ascending=False)
print('Features with missing values (% missing):')
print(missing.round(1).to_string())

Features with missing values (% missing):
PRCP            8.5
HTDD            8.0
CLDD            7.8
TAVG            7.7
TMIN            7.4
DT32            7.4
EMNT            7.4
DX90            7.3
TMAX            7.3
EMXT            7.3
DX70            7.3
DX32            7.3
CASTHMA         4.9
STATION         0.7
ELEVATION       0.7
LONGITUDE       0.7
LATITUDE        0.7
STATION_NAME    0.7


In [7]:
df = df.dropna(subset=["CASTHMA"])

# Asthma

CSATHMA

In [8]:

X = df.drop(target, axis = 1)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')


Train: 5057 samples
Test:  1265 samples


In [22]:
def make_preprocessor():
    """Returns a fresh ColumnTransformer for the Ames dataset."""
    return ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
             categorical_features),
            ('num', SimpleImputer(strategy='mean'),
             numerical_features)
        ]
    )
    # Note: no remainder='passthrough' — all columns are explicitly listed

In [24]:
def create_krr_pipeline(alpha, gamma):
    return make_pipeline(
        make_preprocessor(),
        StandardScaler(),
        KernelRidge(alpha=alpha, kernel='rbf', gamma=gamma)
    )


In [25]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
alphas_krr = np.logspace(-8, -1, num=10)
gammas_krr = np.logspace(-8, -1, num=10)

mean_mses_2d = np.zeros((len(alphas_krr), len(gammas_krr)))

for i, alpha in enumerate(alphas_krr):
    for j, gamma in enumerate(gammas_krr):
        model = create_krr_pipeline(alpha, gamma)
        scores = cross_val_score(
            model, X_train, y_train,
            cv=kf, scoring=make_scorer(mean_squared_error)
        )
        mean_mse = scores.mean()
        mean_mses_2d[i, j] = mean_mse

best_i, best_j = np.unravel_index(np.argmin(mean_mses_2d), mean_mses_2d.shape)
best_alpha_krr = alphas_krr[best_i]
best_gamma_krr = gammas_krr[best_j]

In [26]:
print(f'Best alpha: {best_alpha_krr}')
print(f'Best gamma: {best_gamma_krr}')
print(f'CV MSE:     {mean_mses_2d[best_i, best_j]:.4f}')

Best alpha: 1e-08
Best gamma: 3.5938136638046254e-07
CV MSE:     0.4682


In [27]:
# Using best gamma and alpha
pipe_krr = create_krr_pipeline(alpha=best_alpha_krr, gamma=best_gamma_krr)
pipe_krr.fit(X_train, y_train)
MSE = mean_squared_error(y_test, pipe_krr.predict(X_test))
print(f'KRR Test MSE: {MSE:.4f}')
MSE_scores["KRR"] = MSE

y_pred = pipe_krr.predict(X_test)
accuracy = r2_score(y_test, y_pred)
print(f"R^2 score KRR: {accuracy}")
R2_scores["KRR"] = accuracy

KRR Test MSE: 0.4898
R^2 score KRR: 0.6760891309836348


# Implementation without ML libraries

Okay, so we will need to implement the following functions, that we normally take from sklearn: 

## Model imitalizing 
- KFold
- Train test split

## Preprocessing 
- ColumnTransformer
- OneHotEncoder
- SimpleImputer
- StandardScaler

## Implementation 
- make_scorer
- KernelRidge(alpha, kernel, gamma)
- cross_val_score(model, X, Y, cv, scoring)
- predict
- fit

## Assessment 
- r2_score
- mean_squared_error

# Coding resources for Model implementation 

- [k-fold cross validation](https://medium.com/@avijit.bhattacharjee1996/implementing-k-fold-cross-validation-from-scratch-in-python-ae413b41c80d)

In [ ]:
class ModelInitializer:
    def __init__(self, data):
        self.data = data 
    
    def kfold_indices(self, k):
        fold_size = len(self.data) // k
        indices = np.arange(len(self.data))
        folds = []
        for i in range(k):
            test_indices = indices[i * fold_size: (i + 1) * fold_size]
            train_indices = np.concatenate([indices[:i * fold_size], indices[(i + 1) * fold_size:]])
            folds.append((train_indices, test_indices))
        return folds

    def create_train_test_split(self, folds):
        for train_indices, test_indices in folds:
            X_train, y_train = X[train_indices], y[train_indices]
            X_test, y_test = X[test_indices], y[test_indices]
        return X_train, X_test, y_train, y_test
    

# Coding resources for Preprocessing functions 

- 

In [ ]:
class Preprocessing:
    def __init__(self, X):
        self.X = X
    
    def OneHotEncoder(self):
        pass 

    def SimpleImputer(self):
        pass 

    def StandardScaler(self):
        pass

    def ColumnTransformer(self):
        pass


# Coding resources for Gaussian kernels

We were using the Gaussian kernel for kernel ridge regression when we chose kernel = 'rbf'. 

- [n-d feature vector using numpy and Euclidean distance](https://github.com/kunjmehta/Medium-Article-Codes/blob/master/gaussian-kernel-regression-from-scratch.ipynb)
- [Medium article for above code](https://towardsdatascience.com/kernel-regression-from-scratch-in-python-ea0615b23918/)
- [Geeks for Geeks on Gaussian kernel](https://www.geeksforgeeks.org/machine-learning/gaussian-kernel/)
- [Indepth python explaination and blog article](https://spatial-dev.guru/2026/02/06/kernel-interpolation-in-python-a-complete-beginners-guide-to-gaussian-rbf-kernels-and-rkhs/)
- [Ridge regression in python useful for class structure](https://github.com/KirtanG/Ridge_Regression_from_Scratch/blob/main/notebook/Multiple_Ridge_Regression_From_Scratch.ipynb)

In [ ]:

class GaussianKernel:
    def __init__(self, X, y, sigma = 1.0):
        self.X = X
        self.y = y
        self.sigma = sigma 
        self.alpha = None

    '''Implement the Gaussian Kernel'''
    def compute_gaussian_kernel(self, z):
        return (1/np.sqrt(2*np.pi))*np.exp(-0.5*z**2)

    '''Calculate weights and return prediction'''
    def predict(self, X):
        kernels = np.array([self.compute_gaussian_kernel((np.linalg.norm(xi-X))/self.b) for xi in self.x])
        weights = np.array([len(self.x) * (kernel/np.sum(kernels)) for kernel in kernels])
        return np.dot(weights.T, self.y)/len(self.x)

class KernelRidgeRegression:
    """
    Implements Kernel Ridge Regression using NumPy.

    This class provides methods to fit a kernel ridge regression model
    to training data and make predictions on new data. It calculates the
    coefficients and intercept using the closed-form solution (Normal Equation).
    """
    def __init__(self,alpha = 0.1, gamma = 0.1):
        """
        Initializes the model.

        Attributes:
            coef_ (np.ndarray): The learned coefficients for the features.
            intercept_ (float): The learned intercept (bias) term.
            alpha: The regularisation term.
        """
        self.alpha = alpha
        self.gamma = gamma
        self.coef_ = None
        self.intercept_ = None

    def fit(self,X_train:np.ndarray,y_train:np.ndarray):
        """
        Fits the multiple ridge regression model to the training data.

        The coefficients and intercept are calculated using the Normal Equation:
        betas = (X_train.T @ X_train)^-1 @ X_train.T @ y_train

        Args:
            X_train (np.ndarray): Training feature data. Can be 1D or 2D.
                                  If 1D, it will be reshaped to 2D.
            y_train (np.ndarray): Training target values.
        """
         # Ensure that the array is 2D even if the array is 1D
        if X_train.ndim == 1:
            X_train = X_train.reshape(-1, 1)
        # Add a column of ones to X_train for the intercept term
        X_train = np.insert(X_train,0,1,axis = 1)
        # Add the identity matrix
        I = np.identity(X_train.shape[1])
        # Set the first row and column's value to zero.
        I[0][0] = 0
        
        
        # Calculate betas (coefficients and intercept) using the Normal Equation
        # np.linalg.inv: computes the inverse of a matrix
        # np.dot: performs dot product of two arrays
        betas = np.linalg.inv(np.dot(X_train.T,X_train) + self.alpha * I).dot(X_train.T).dot(y_train)
        
        # get the intercept from the matrix
        self.intercept_ = betas[0] 
        self.coef_ = betas[1:]

    def predict(self,X_test:np.ndarray) -> np.ndarray:
        """
        Predicts target values for new data using the trained model.

        Args:
            X_test (np.ndarray): Feature data for making predictions. Can be 1D or 2D.
                                 If 1D, it will be reshaped to 2D.

        Returns:
            np.ndarray: Predicted target values.
        """
        is_1D = False 
        if X_test.ndim == 1:
            X_test = X_test.reshape(-1,1)
            # Set the flag to True if the input array is 1D
            is_1D = True 
        if self.coef_.ndim == 1:
            self.coef_ = self.coef_.reshape(-1,1)
        y_pred = np.dot(X_test,self.coef_) + self.intercept_
        if is_1D:
            # Flatten the array so as to match the original array
            y_pred = y_pred.flatten() 
        return y_pred 
        

# Coding resources for assessment 